# Average Metric Repair — Experiments (pure Python)

Uses the pure-Python library (`graph_models.py`, `metric_repair.py`) and the importable harness in
`run_experiments.py`. No Sage. Run this with a normal Python 3 kernel that has numpy / scipy /
networkx / pandas.

The harness is the same one the cluster CLI uses, so what you validate here is exactly what runs at
scale. `run_sweep` parallelises **across instances** (each algorithm timed in isolation, so
`runtime_s` stays comparable); serial and parallel give identical results.

## Setup

In [ ]:
import numpy as np
import pandas as pd

from run_experiments import (build_tasks, run_sweep, summarize, per_instance, save_results,
                             run_instance, GENERATORS, ALGORITHMS)
from graph_models import seed_all, random_geometric_weighted_graph, random_uniform_weighted_graph
from metric_repair import make_index_encoding, get_weights_vector, induced_cycle_matrix

print("generators:", list(GENERATORS))
print("algorithms:", [(n, v) for n, v, _, _ in ALGORITHMS])

## Cover-size experiments

`build_tasks` builds the sweep grid; `run_sweep` runs it (parallel across instances) and returns one
tidy row per (instance, component, algorithm). Aggregate with `summarize`. Uncomment `save_results`
to persist a git-stamped `.csv.gz` into `results/`.

In [ ]:
# sweep over n at p = 0.3
tasks_n = build_tasks(["geometric"], ns=[40, 60, 80], p_of_n=lambda n: 0.3, trials=5, base_seed=0)
df_n = run_sweep(tasks_n, parallel=True)
print(f"{len(df_n)} rows | all covers valid: {bool((df_n['valid'] == 1).all())}")
# save_results(df_n, "alg_sweep_n")
summarize(df_n)

In [ ]:
# sweep over p at n = 80
tasks_p = build_tasks(["geometric"], ns=[80], ps=list(np.linspace(0.1, 0.9, 5)), trials=5, base_seed=1000)
df_p = run_sweep(tasks_p, parallel=True)
print(f"{len(df_p)} rows | all covers valid: {bool((df_p['valid'] == 1).all())}")
# save_results(df_p, "alg_sweep_p")
summarize(df_p)

## Threshold experiments (broken-cycle probability)

A separate family: how often a random graph has a broken induced cycle, as a function of
`p = N^{-q}`. Translated faithfully from the Sage notebook (`phi @ w < 0` flags a broken chordless
cycle, via `induced_cycle_matrix`). Each is reproducible from its `seed`.

In [ ]:
def experiment_demonstrate_threshold(N=30, AVG=30, ps=20, seed=0):
    """Mean number of broken induced cycles vs p = N^-q (geometric weights). res[0]=broken, res[1]=#cycles."""
    seed_all(seed)
    delta = 1.9 / ps
    P = [2 - i * delta for i in range(ps)]
    res = np.zeros((2, ps))
    for j, q in enumerate(P):
        print(f"finished {100 * j // ps}%")
        p = 1 / np.power(N, q)
        tot = 0
        for _ in range(AVG):
            G = random_geometric_weighted_graph(N, p)
            while G.number_of_edges() == 0:                # make sure the graph has edges
                G = random_geometric_weighted_graph(N, p)
            w = get_weights_vector(G, make_index_encoding(G))
            phi, count = induced_cycle_matrix(G)
            V = phi @ w                                    # potential deficit per cycle row
            tot += np.count_nonzero(V < 0)                 # broken cycles
        res[0, j] = tot / AVG
        res[1, j] = count
    return res


def experiment_demonstrate_threshold_prob(N=30, AVG=30, ps=50, std=False, seed=0):
    """Probability a random graph has a broken cycle (and has any cycle) vs p = N^-q."""
    seed_all(seed)
    delta = 1.9 / ps
    P = [2 - i * delta for i in range(ps)]
    res = np.zeros((2, ps))
    res_std = np.zeros((2, ps))
    for j, q in enumerate(P):
        print(f"finished {100 * j // ps}%")
        p = 1 / np.power(N, q)
        samples_broken = np.zeros(AVG)
        samples_cycles = np.zeros(AVG)
        for trial in range(AVG):
            G = random_geometric_weighted_graph(N, p)
            while G.number_of_edges() == 0:
                G = random_geometric_weighted_graph(N, p)
            w = get_weights_vector(G, make_index_encoding(G))
            phi, count = induced_cycle_matrix(G)
            V = phi @ w
            samples_cycles[trial] = 1 if count > 0 else 0
            samples_broken[trial] = 1 if np.count_nonzero(V < 0) > 0 else 0
        res[0, j] = samples_broken.mean()
        res[1, j] = samples_cycles.mean()
        res_std[0, j] = np.std(samples_broken) / np.sqrt(AVG)
        res_std[1, j] = np.std(samples_cycles) / np.sqrt(AVG)
    return (res, res_std) if std else res

In [ ]:
# quick threshold demo (small AVG/ps so it runs fast)
res = experiment_demonstrate_threshold_prob(N=30, AVG=20, ps=8, seed=0)
print("P(broken cycle):", np.round(res[0], 2))
print("P(has cycle)   :", np.round(res[1], 2))